# G1 Academy 4 - Solution: SLAM, perception, and delivery


## Introduction
SLAM lifecycle: start mapping, capture external visualization, stop/save, relocate, save named poses, then navigate only with valid localization. The instructor-supplied SLAM adapter exposes the deployed native operations. The academy supplies perception, pose estimation, wrist-palm calibration, and IK solvers; util.load_provided_pipeline loads those heavy components.

Serial arm placeholder: base -- shoulder -- elbow -- wrist -- palm. Each joint affects all later links; the supplied IK solver converts a target pose/increment into bounded joint motion.

## Provided SLAM, perception, pose, and IK pipelines

This academy does not require participants to install or reverse-engineer an internal SLAM implementation. The instructor supplies one configured SLAM adapter object for the installed robot/map-service version. It exposes the small stable interface used in this notebook: status(), start_mapping(), stop_mapping(path), relocate(path), save_named_pose(name), navigate_to(name), and, where supported, navigate_path(names). The adapter owns its DDS/RPC boilerplate; participants learn the operational order, validity checks, and error handling.

The supplied perception pipeline accepts a current RGB-D frame and task prompt, returns detections/segmentation with confidence and frame metadata, and can estimate an ArUco or object pose. The supplied pose layer transforms camera output into robot/base coordinates and applies the calibrated wrist-to-palm offset. The supplied IK executor accepts that calibrated end-effector target, enforces joint/velocity limits, and moves in bounded increments. Participants must validate freshness, confidence, calibration availability, and resulting motion; they do not implement PCA, ArUco solving, transforms, or inverse kinematics from scratch.

Mapping workflow: start mapping, move safely while collecting an external visualization, stop/save, relocate against the map, verify localization, then save named pickup/dropdown poses. A delivery workflow: acquire RGB-D, perceive target, validate pose, open hand, approach with incremental IK, close and verify grasp, move to the supplied stable-hold pose, navigate, re-perceive drop target, move incrementally, open, retreat, and release.

Serial manipulator placeholder:

    base -- shoulder -- elbow -- wrist -- palm/end effector
             q1          q2       q3..q7

Every joint changes the pose of each later link, which is why the supplied IK result must be bounded and observed rather than sent as one uncontrolled jump.

## slam_util.py: what it owns and what participants own

slam_util.py is local academy code, not an unknown external dependency. It directly creates the Unitree ChannelFactory, subscribes to rt/slam_info and rt/slam_key_info, registers the SLAM RPC API IDs, sends mapping/relocation/pose-navigation requests, parses current poses, persists named points in JSON, waits for arrival, and records the path to an externally captured map visualization.

Create SlamUtility once for one interface/domain. Call status() before operations; start_mapping(), stop_mapping(), and relocate() return normalized result dictionaries with ok/code/raw. save_named_pose(name) requires a fresh pose. navigate_to(name) requires successful relocation. view_map() returns only an existing external visualization artifact registered with set_visualization_path(path).

The deployed RPC interface known from nav_bot supports single pose navigation. navigate_path deliberately requires a verified native_path_callback. This prevents the misleading point-by-point fallback that would violate the course requirement for built-in complete-path navigation.

The provided perception pipeline is separate. It offers RGB-D detection/segmentation, confidence/frame metadata, ArUco or object pose estimation, camera-to-base transforms, calibrated wrist-palm offset, and bounded incremental IK. Participants validate inputs/outputs and orchestrate the sequence; they do not implement those algorithms.

## Concrete IK increment implementation

util.make_recognition_ik_increment(robot, arm) creates the actual callback used by DeliveryPipeline. It follows recognition_app_v3: ArmExecutor reads the current arm joints from robot.get_joint_states(), ArmFK computes the current end-effector transform, a translation increment is applied, ArmIK solves DLS IK from that current pose, then ArmExecutor validates reachability and interpolates a rate-limited joint trajectory through robot.move_upper_body_joint(...).

The robot argument must be the recognition stack's sdk_client.Robot, or a direct-SDK adapter exposing those two methods. The callback returns success/reason, IK diagnostics, and target translation. DeliveryPipeline should use incrementer.current_palm_xyz() as its start feedback. It is no longer an unspecified callback placeholder.


## Task 1 - Use native SLAM client/subscriber
SlamInfoSubscriber observes status. SlamOperateClient performs RPC operations. Follow the nav_bot reference sequencing; do not publish raw SLAM control messages.


In [ ]:
from slam_util import SlamUtility
slam = SlamUtility(interface="eth0", domain_id=0, map_path="/home/unitree/test.pcd")
def mapping_status():
    return slam.status()
def start_mapping():
    return slam.start_mapping("indoor")
def stop_mapping():
    return slam.stop_mapping()
def relocate():
    return slam.relocate()
# print(mapping_status())


## Task 2 - Build named-point navigation
Keep named points in an application dictionary/file. Add only after valid localization. Confirm the deployed client supports native complete-path navigation before implementing navigate_path; otherwise document the limitation rather than pretending sequential calls are a path API.


In [ ]:
def add_point(name):
    return slam.save_named_pose(name)
def remove_point(name):
    return slam.remove_named_pose(name)
def navigate_to_point(name):
    return slam.navigate_to(name)
def view_map():
    return slam.view_map()
def navigate_path(names, native_path_callback):
    return slam.navigate_path(names, native_path_callback)


## Task 3 - Use the local DeliveryPipeline

DeliveryPipeline in util.py contains the tedious OpenAI image request, ArUco pose estimation, camera/base and wrist/palm transforms, and small-step IK orchestration. Supply calibration matrices from the academy calibration file and a bounded direct IK increment function from the arm-control lesson. Validate perception confidence and frame freshness before calling the motion portion.


In [ ]:
from util import DeliveryPipeline, make_recognition_ik_increment
def make_delivery_motion(robot, camera_matrix, distortion, camera_to_base, wrist_to_palm, arm="right"):
    pipeline = DeliveryPipeline(camera_matrix, distortion, camera_to_base, wrist_to_palm)
    ik_increment = make_recognition_ik_increment(robot, arm=arm)
    return pipeline, ik_increment
def approach_marker(robot, bgr_frame, marker_length_m, calibration, arm="right"):
    pipeline, increment = make_delivery_motion(robot, *calibration, arm=arm)
    target = pipeline.marker_to_palm_target(pipeline.aruco_pose(bgr_frame, marker_length_m))
    return pipeline.execute_incremental_ik(increment, increment.current_palm_xyz(), target, side=arm)
# robot is the recognition stack sdk_client.Robot or a compatible direct-SDK adapter.


### Safety
Run no command cell until the subscriber state is fresh, controller ownership is known, the space is clear, and a damp path is available. Code is not invoked automatically.
